In [18]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorWithPadding, TrainingArguments, Trainer, glue_compute_metrics
from datasets import load_dataset

## 一、加载数据集
下载数据集到指定目录
```bash
huggingface-cli download \
--repo-type dataset \
--resume-download cornell-movie-review-data/rotten_tomatoes \
--local-dir-use-symlinks False \
--local-dir /Users/wangweijun/LLM/datasets/cornell-movie-review-data/rotten_tomatoes
```

In [3]:
# 数据集名称
DATASET_NAME = "/Users/wangweijun/LLM/datasets/cornell-movie-review-data/rotten_tomatoes"

# 加载数据集
raw_datasets = load_dataset(DATASET_NAME)
print(f"===> 数据集加载完成\n{raw_datasets}")

# 训练集
raw_train_dataset = raw_datasets["train"]
# 验证集
raw_valid_dataset = raw_datasets["validation"]
# 测试集
raw_test_dataset = raw_datasets["test"]

===> 数据集加载完成
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


## 二、加载模型
下载模型到指定目录
```bash
huggingface-cli download \
--resume-download openai-community/gpt2 \
--local-dir-use-symlinks False \
--local-dir /Users/wangweijun/LLM/models/openai-community/gpt2
```

In [4]:
# 模型名称
MODEL_NAME = "/Users/wangweijun/LLM/models/openai-community/gpt2"

# 加载模型
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f"===> 模型加载完成\n{model}")

===> 模型加载完成
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


## 三、加载分词器

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
tokenizer.pad_token_id = 0
print(f"===> Tokenizer 加载完成\n{tokenizer}")

===> Tokenizer 加载完成
GPT2TokenizerFast(name_or_path='/Users/wangweijun/LLM/models/openai-community/gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '!'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50257: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## 四、其他相关公共变量赋值

In [6]:
# 设置随机种子，同个种子的随机序列可复现
transformers.set_seed(42)

# 标签集
named_labels = ["neg", "pos"]

# 标签转 token_id
label_ids = [tokenizer(named_labels[i], add_special_tokens=False)["input_ids"][0] for i in range(len(named_labels))]
print(f"===> 标签转 token_id\n{label_ids}")

===> 标签转 token_id
[12480, 1930]


## 五、处理的数据集
- 模型的输入：<提示词...><输入文本><提示词...>
- 模型输出：<输入文本>
- 文本转 Token_IDs：<PROMPT_TOKEN_IDS><INPUT_TOKEN_IDS><PROMPT_TOKEN_IDS><OUTPUT_TOKEN_IDS>
- PAD 成相等长度
    - `<INPUT_1_1><OUTPUT_1_1><INPUT_1_2><OUTPUT_1_2><PAD>...<PAD>`
    - `<INPUT_2_1><OUTPUT_2_1><INPUT_2_2><OUTPUT_2_2><PAD>...<PAD>`
- 标识出不参与 Attention 计算的 Tokens（Attention Mask）
    - `<1><1>...<1><1><0><0>...<0>`
- 标识出参与 loss 计算的 Tokens（只有输出 Tokens 才参与 loss 计算）
    - `<-100><-100>...<-100><OUTPUT_TOKEN_IDS><-100>...<-100>`

<div class="alert alert-block alert-info">
划重点：实际带入模型计算的是三个矩阵（tensor） <br>
1. 经过拼接和 PADDING 的输入 Token 序列<br>
2. Attention Mask 序列，标识出 1 中有效的 Tokens（用于 Attention 计算）<br>
3. Labels 序列，标识出 1 中作为输入的Tokens（用于 loss 计算）<br>
</div>

In [7]:
# 最大序列长度（输入序列长度 + 输出序列长度）
MAX_LEN=512
# 数据集的输入字段名称
DATA_BODY_KEY="text"
# 数据集的标签字段名称
DATA_LABEL_KEY="label"

# 定义数据处理函数，把原始数据转换成: {input_ids, attention_mask, labels}
def process_fn(examples):
    model_inputs = {
        "input_ids": [],
        "attention_mask": [],
        "labels": [],
    }
    for i in range(len(examples[DATA_BODY_KEY])):
        # 自定义 Prompt 格式
        prompt = f"{examples[DATA_BODY_KEY][i]} Sentiment: "
        inputs = tokenizer(prompt, add_special_tokens=False)
        label = label_ids[examples[DATA_LABEL_KEY][i]]
        input_ids = inputs["input_ids"] + [label]

        raw_len = len(input_ids)

        if raw_len >= MAX_LEN:
            input_ids = input_ids[-MAX_LEN:]
            attention_mask = [1] * MAX_LEN
            labels = [-100] * (MAX_LEN - 1) + [label]
        else:
            input_ids = input_ids + [tokenizer.pad_token_id] * (MAX_LEN - raw_len)
            attention_mask = [1] * raw_len + [0] * (MAX_LEN - raw_len)
            labels = [-100] * (raw_len - 1) + [label] + [-100] * (MAX_LEN - raw_len)

        model_inputs["input_ids"].append(input_ids)
        model_inputs["attention_mask"].append(attention_mask)
        model_inputs["labels"].append(labels)

    return model_inputs

In [11]:
# 处理训练集
tokenized_train_dataset = raw_train_dataset.map(
    function=process_fn,
    batched=True,
    # 表示删除原始数据集中的所有列，只保留 process_fn 函数生成的新列
    remove_columns=raw_train_dataset.column_names,
    desc="Running tokenizer on train dataset",
)
print(f"===> 处理训练集完成\n{tokenized_train_dataset}")

# 处理验证集
tokenized_valid_dataset = raw_valid_dataset.map(
    function=process_fn,
    batched=True,
    # 表示删除原始数据集中的所有列，只保留 process_fn 函数生成的新列
    remove_columns=raw_valid_dataset.column_names,
    desc="Running tokenizer on valid dataset",
)
print(f"===> 处理验证集完成\n{tokenized_valid_dataset}")

===> 处理训练集完成
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8530
})
===> 处理验证集完成
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1066
})


## 六、定义数据规整器
训练时自动将数据拆分成 batch

In [14]:
# 定义数据校准器（自动生成 batch）
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

## 七、定义训练超参数
比如，学习率
https://huggingface.co/docs/transformers/v4.52.2/en/main_classes/trainer#transformers.TrainingArguments

In [17]:
# 学习率
LR=2e-5
# 批次大小
BATCH_SIZE=8
# 每多少步，输出一次 log/做一次 eval
INTERVAL=100

training_args = TrainingArguments(
    # checkpoint 保存路径
    output_dir="./output",
    # 如果输出目录（output_dir）已存在，是否覆盖其内容
    overwrite_output_dir=True,
    # steps：按固定步数评估（需配合 eval_steps）
    # epoch：每个训练周期结束后评估
    # no：不进行评估。
    eval_strategy="steps",
    # 指定训练的总轮数
    num_train_epochs=1,
    # 每个设备（GPU/CPU）在训练时的批量大小
    per_device_train_batch_size=BATCH_SIZE,
    # 梯度累积步数。将多个小批次的梯度累积后再更新模型，等效于使用更大的批次
    # gradient_accumulation_steps=4 且 per_device_train_batch_size=8 时，实际批量大小为 8×4=32
    gradient_accumulation_steps=1,
    # 每个设备在评估时的批次大小
    per_device_eval_batch_size=BATCH_SIZE,
    # 当 evaluation_strategy="steps" 时，指定每隔多少步进行一次评估
    eval_steps=INTERVAL,
    # 每多少步，保存一次检查点的间隔
    save_steps=INTERVAL,
    # 学习率
    learning_rate=LR,
)

## 八、定义训练器
https://huggingface.co/docs/transformers/main_classes/trainer

In [22]:
# 节省显存
model.gradient_checkpointing_enable()

trainer = Trainer(
    # 待训练的模型
    model=model,
    # 训练参数
    args=training_args,
    # 训练数据集
    train_dataset=tokenized_train_dataset,
    # 评估数据集
    eval_dataset=tokenized_valid_dataset,
    # 数据校准器
    data_collator=collator,
    # 计算自定义评估指标
    # compute_metrics=glue_compute_metrics,
)

## 九、开始训练

In [23]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,No log,0.602716
200,No log,0.494007
300,No log,0.375975
400,No log,0.386066
500,0.717300,0.419605
600,0.717300,0.357083
700,0.717300,0.388153
800,0.717300,0.372506
900,0.717300,0.344667
1000,0.428300,0.342399


TrainOutput(global_step=1067, training_loss=0.5632092923680084, metrics={'train_runtime': 1203.1285, 'train_samples_per_second': 7.09, 'train_steps_per_second': 0.887, 'total_flos': 2228821032960000.0, 'train_loss': 0.5632092923680084, 'epoch': 1.0})

## 十、使用 tensorboard 工具可视化训练过程（可选）
在系统命令行模式下执行：`tensorboard --logdir ./output`

In [ ]:
# !pip install tensorboard

%load_ext tensorboard
%tensorboard --logdir ./output

## 十一、加载训练后的模型进行推理
checkpoint 指的是在特定的时间点保存的模型权重状态，这个状态包括了模型的参数权重和优化器状态，使得训练可以从这个点重新开始，而不是重新开始训练。<br>

通常，我们通过观察在验证集傻姑娘点评估结果，选择某个 checkpoint 作为最终用于推理的模型。<br>

In [44]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 加载训练后的权重
model = AutoModelForCausalLM.from_pretrained("./output/checkpoint-1067")

# 模型转为评估模式
# 某些层在训练和推理时的行为不同：
# Dropout：训练时随机丢弃神经元以防止过拟合，但推理时需保留所有神经元。
# BatchNorm：训练时使用当前批次的统计信息（均值、方差），推理时使用训练阶段累积的全局统计信息。
# 如果不设置 model.eval()，这些层会继续使用训练时的行为，导致推理结果不稳定或不准确。
model.eval()

# 加载 tokenizer
MODEL_NAME = "/Users/wangweijun/LLM/models/openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# 待分类文本
text = "I really enjoyed the movie. It was a great experience."

# 输入文本转换为 token IDs
# 注意：提示词格式要和训练时一致
inputs = tokenizer(f"{text} Sentiment: ", return_tensors="pt")

# 推理：预测标签
outputs = model.generate(**inputs, do_sample=False, max_new_tokens=1)

# 输出结果
print(outputs)
tokenizer.decode(outputs[0][-1])


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


tensor([[   40,  1107,  8359,   262,  3807,    13,   632,   373,   257,  1049,
          1998,    13, 11352,  3681,    25,   220,  1930]])


'pos'

## 十二、加载 checkpoint 继续训练（可选）
resume_from_checkpoint 参数指定了要从中恢复训练的 checkpoint 路径。<br>

In [ ]:
trainer.train(resume_from_checkpoint="./output/checkpoint-1067")


# 学习率
LR=2e-5
# 批次大小
BATCH_SIZE=8
# 每多少步，输出一次 log/做一次 eval
INTERVAL=100

training_args = TrainingArguments(
    # checkpoint 保存路径
    output_dir="./output",
    # 如果输出目录（output_dir）已存在，是否覆盖其内容
    overwrite_output_dir=True,
    # steps：按固定步数评估（需配合 eval_steps）
    # epoch：每个训练周期结束后评估
    # no：不进行评估。
    eval_strategy="steps",
    # 指定训练的总轮数
    num_train_epochs=1,
    # 每个设备（GPU/CPU）在训练时的批量大小
    per_device_train_batch_size=BATCH_SIZE,
    # 梯度累积步数。将多个小批次的梯度累积后再更新模型，等效于使用更大的批次
    # gradient_accumulation_steps=4 且 per_device_train_batch_size=8 时，实际批量大小为 8×4=32
    gradient_accumulation_steps=1,
    # 每个设备在评估时的批次大小
    per_device_eval_batch_size=BATCH_SIZE,
    # 当 evaluation_strategy="steps" 时，指定每隔多少步进行一次评估
    eval_steps=INTERVAL,
    # 每多少步，保存一次检查点的间隔
    save_steps=INTERVAL,
    # 学习率
    learning_rate=LR,
)

# 节省显存
model.gradient_checkpointing_enable()

trainer = Trainer(
    # 待训练的模型
    model=model,
    # 训练参数
    args=training_args,
    # 训练数据集
    train_dataset=tokenized_train_dataset,
    # 评估数据集
    eval_dataset=tokenized_valid_dataset,
    # 数据校准器
    data_collator=collator,
    # 计算自定义评估指标
    # compute_metrics=glue_compute_metrics,
)

trainer.train()


<div class="alert alert-success">
<b>扩展知识</b></br>
<li>以上实验是从生成模型的视角，训练一个文本分类任务</li>
<li>上述的分类任务，也可以从分类器的视角进行建模，使用类似 BERT 加额外的分类器层的形式实现</li>
</div>

## 总结上述过程
1. 加载数据集
2. 数据预处理
    - 将输入、输出按照特定格式拼接
    - 文本转 Token_IDs
    - 通过 labels 标识出那部分是输出（只有输出的 token 参与 loss 计算）
3. 加载模型、Tokenizer
4. 定义数据规整器
5. 定义训练超参：学习率、批次大小、训练轮数、评估间隔等
6. 定义训练器
7. 开始训练
8. 注意：训练后推理时，输入数据的拼接方式要和训练时保持一致
